# Task 11 — End-to-End Validation

**Purpose:** Assert that all 4 pipeline tasks ran successfully and produced correct, consistent output across every layer.

| Layer | Tables validated |
|---|---|
| Bronze | bronze_metadata |
| Silver | silver_metadata, bronze_metadata_quarantine, 9 entity/bridge tables |
| Gold | gold_metadata, gold_certified_metadata_final, gold_quality_metrics_final |
| Monitor | pipeline_run_log |

**Hard invariant:** `silver_rows + quarantine_rows == bronze_rows == 10 000`

## Step 1 — Bronze Layer Validation

In [ ]:
from pyspark.sql.functions import col, count, when, isnull, current_date
from datetime import datetime, timezone, timedelta

EXPECTED_BRONZE_ROWS = 10_000
MIN_COMPLIANCE_SCORE = 0.0   # overall catalog score must be above this
FRESHNESS_HOURS      = 25    # tables must have been written within this window

print("=" * 65)
print("STEP 1 — BRONZE LAYER")
print("=" * 65)

bronze_df   = spark.table("metadata_governance.bronze.bronze_metadata")
bronze_rows = bronze_df.count()

required_audit_cols = ["ingestion_timestamp", "load_date", "record_source"]
missing_audit = [c for c in required_audit_cols if c not in bronze_df.columns]

null_record_source = bronze_df.filter(
    isnull(col("record_source")) | (col("record_source") == "")
).count()

print(f"  Row count          : {bronze_rows}  (expected {EXPECTED_BRONZE_ROWS})")
print(f"  Audit cols present : {required_audit_cols}")
print(f"  Missing audit cols : {missing_audit if missing_audit else 'none'}")
print(f"  Null record_source : {null_record_source}")

assert bronze_rows == EXPECTED_BRONZE_ROWS, \
    f"E2E FAIL — bronze_metadata row count: expected {EXPECTED_BRONZE_ROWS}, got {bronze_rows}"
assert not missing_audit, \
    f"E2E FAIL — missing audit columns: {missing_audit}"
assert null_record_source == 0, \
    f"E2E FAIL — {null_record_source} rows have null record_source"

print("  PASS — Bronze layer validated")
print("=" * 65)

## Step 2 — Silver Layer Validation

In [ ]:
print("=" * 65)
print("STEP 2 — SILVER LAYER")
print("=" * 65)

silver_df       = spark.table("metadata_governance.silver.silver_metadata")
quarantine_df   = spark.table("metadata_governance.silver.bronze_metadata_quarantine")
silver_rows     = silver_df.count()
quarantine_rows = quarantine_df.count()
silver_total    = silver_rows + quarantine_rows

print(f"  silver_metadata rows              : {silver_rows}")
print(f"  bronze_metadata_quarantine rows   : {quarantine_rows}")
print(f"  Combined total                    : {silver_total}  (must equal {EXPECTED_BRONZE_ROWS})")

# Key columns must not be null in silver
null_table_name = silver_df.filter(
    isnull(col("table_name")) | (col("table_name") == "")
).count()

print(f"  Null table_name in silver         : {null_table_name}  (must be 0)")

# 9 entity tables must exist and be non-empty
entity_tables = [
    "dim_system", "dim_database", "dim_schema", "dim_table", "dim_column",
    "dim_term", "dim_tag", "bridge_col_term", "bridge_col_tag"
]
print("\n  Entity table row counts:")
entity_results = {}
for t in entity_tables:
    cnt = spark.table(f"metadata_governance.silver.{t}").count()
    entity_results[t] = cnt
    status = "OK" if cnt > 0 else "EMPTY"
    print(f"    {status}  {t:<22} : {cnt} rows")

assert silver_total == EXPECTED_BRONZE_ROWS, \
    f"E2E FAIL — silver + quarantine = {silver_total}, expected {EXPECTED_BRONZE_ROWS}"
assert silver_rows > 0, \
    "E2E FAIL — silver_metadata is empty"
assert null_table_name == 0, \
    f"E2E FAIL — {null_table_name} silver rows have null table_name"
empty_entities = [t for t, c in entity_results.items() if c == 0]
assert not empty_entities, \
    f"E2E FAIL — empty entity tables: {empty_entities}"

print("\n  PASS — Silver layer validated")
print("=" * 65)

## Step 3 — Gold Layer Validation

In [ ]:
print("=" * 65)
print("STEP 3 — GOLD LAYER")
print("=" * 65)

gold_base_df  = spark.table("metadata_governance.gold.gold_metadata")
gold_cert_df  = spark.table("metadata_governance.gold.gold_certified_metadata_final")
gold_metrics_df = spark.table("metadata_governance.gold.gold_quality_metrics_final")

gold_base_rows  = gold_base_df.count()
gold_cert_rows  = gold_cert_df.count()
gold_metrics_rows = gold_metrics_df.count()

pass_count = gold_cert_df.filter(col("certification_status") == "PASS").count()
fail_count = gold_cert_df.filter(col("certification_status") == "FAIL").count()
overall_score = (pass_count / gold_cert_rows * 100) if gold_cert_rows > 0 else 0

# Verify completeness_score column exists and has no nulls
null_scores = gold_cert_df.filter(isnull(col("completeness_score"))).count()

# Verify 8 rules have metrics rows
rule_metrics_count = gold_metrics_df.filter(col("level") == "rule").count()

print(f"  gold_metadata rows                : {gold_base_rows}")
print(f"  gold_certified_metadata_final rows: {gold_cert_rows}  (must equal {EXPECTED_BRONZE_ROWS})")
print(f"  gold_quality_metrics_final rows   : {gold_metrics_rows}")
print(f"  Certified PASS                    : {pass_count}")
print(f"  Certified FAIL                    : {fail_count}")
print(f"  Overall compliance score          : {overall_score:.2f}%")
print(f"  Null completeness_score rows      : {null_scores}")
print(f"  Rule-level metric rows            : {rule_metrics_count}  (expected 8)")

assert gold_cert_rows == EXPECTED_BRONZE_ROWS, \
    f"E2E FAIL — gold_certified rows: expected {EXPECTED_BRONZE_ROWS}, got {gold_cert_rows}"
assert gold_metrics_rows > 0, \
    "E2E FAIL — gold_quality_metrics_final is empty"
assert overall_score > MIN_COMPLIANCE_SCORE, \
    f"E2E FAIL — compliance score {overall_score:.2f}% is below minimum {MIN_COMPLIANCE_SCORE}%"
assert null_scores == 0, \
    f"E2E FAIL — {null_scores} rows have null completeness_score"
assert rule_metrics_count == 8, \
    f"E2E FAIL — expected 8 rule metric rows, got {rule_metrics_count}"

print("\n  PASS — Gold layer validated")
print("=" * 65)

## Step 4 — Pipeline Log Validation

In [ ]:
from pyspark.sql.functions import max as spark_max

print("=" * 65)
print("STEP 4 — PIPELINE LOG")
print("=" * 65)

log_df   = spark.table("metadata_governance.default.pipeline_run_log")
log_rows = log_df.count()

latest_ts = log_df.agg(spark_max("run_timestamp").alias("latest")).collect()[0]["latest"]
freshness_cutoff = datetime.now(timezone.utc) - timedelta(hours=FRESHNESS_HOURS)

# Check all 3 layers were logged in the latest run
layers_logged = [
    r["layer"]
    for r in log_df.filter(col("run_timestamp") == latest_ts)
                   .select("layer")
                   .collect()
]

print(f"  Total log entries   : {log_rows}")
print(f"  Latest run          : {latest_ts}")
print(f"  Freshness cutoff    : {freshness_cutoff}  ({FRESHNESS_HOURS}h window)")
print(f"  Layers in last run  : {sorted(layers_logged)}")

assert log_rows > 0, \
    "E2E FAIL — pipeline_run_log is empty"
assert latest_ts is not None, \
    "E2E FAIL — no run_timestamp found in log"
for layer in ["bronze", "silver", "gold"]:
    assert layer in layers_logged, \
        f"E2E FAIL — '{layer}' layer not logged in the latest run"

print("\n  PASS — Pipeline log validated")
print("=" * 65)

## Step 5 — Cross-Layer Record Count Invariant

In [ ]:
print("=" * 65)
print("STEP 5 — CROSS-LAYER INVARIANT")
print("=" * 65)
print(f"  Bronze rows                       : {bronze_rows}")
print(f"  Silver valid + quarantine         : {silver_rows} + {quarantine_rows} = {silver_total}")
print(f"  Gold certified (all records)      : {gold_cert_rows}")
print()

assert bronze_rows == silver_total, \
    f"E2E FAIL — bronze ({bronze_rows}) != silver+quarantine ({silver_total})"
assert bronze_rows == gold_cert_rows, \
    f"E2E FAIL — bronze ({bronze_rows}) != gold_certified ({gold_cert_rows})"

print("  PASS — Record counts are consistent across all three layers")
print("=" * 65)

## Step 6 — End-to-End Summary

In [ ]:
print("=" * 65)
print("END-TO-END VALIDATION — FINAL SUMMARY")
print("=" * 65)
summary = [
    ("Bronze rows = 10 000",                bronze_rows == EXPECTED_BRONZE_ROWS),
    ("Bronze audit columns present",        not missing_audit),
    ("Silver + quarantine = 10 000",        silver_total == EXPECTED_BRONZE_ROWS),
    ("No null table_name in Silver",        null_table_name == 0),
    ("All 9 entity tables populated",       not empty_entities),
    ("Gold certified rows = 10 000",        gold_cert_rows == EXPECTED_BRONZE_ROWS),
    ("Compliance score > 0%",               overall_score > MIN_COMPLIANCE_SCORE),
    ("No null completeness_score",          null_scores == 0),
    ("8 rule-level metrics rows exist",     rule_metrics_count == 8),
    ("Pipeline log has entries",            log_rows > 0),
    ("All 3 layers logged in latest run",   all(l in layers_logged for l in ["bronze","silver","gold"])),
    ("Cross-layer count invariant holds",   bronze_rows == silver_total == gold_cert_rows),
]
all_pass = True
for label, result in summary:
    status = "PASS" if result else "FAIL"
    if not result:
        all_pass = False
    print(f"  {status}  {label}")
print("=" * 65)
print(f"  Overall compliance score : {overall_score:.2f}%")
print(f"  PASS / FAIL              : {pass_count} / {fail_count}")
print("=" * 65)
if all_pass:
    print("  ALL END-TO-END CHECKS PASSED — pipeline is production-ready")
else:
    raise AssertionError("One or more end-to-end checks failed — see above")
print("=" * 65)